Копия `Old/BDT_rho_pi.ipynb`, приведённая к схеме `BDT_rho_pi_lep.ipynb`.

FoM = εS / (a/2 + √((Nфон + 1)/3)), a = 3. Фон: части 1–3 `gen_data/`.

Поле `FoM` в JSON модели содержит пороги для совместимости. Численные максимумы и кривые сохранены отдельно. Нормировка фона и truth-отбор `is0 == 1` требуют отдельного физического обоснования.


## 1. Импорты и оформление


In [ ]:
from bdt_header import (
    json, Path, np, pd, ds, plt, PLOT_COLORS,
    train_test_split, XGBClassifier,
    classification_report, roc_auc_score, confusion_matrix,
    get_values, errordot, json_update,
    mask_physics, mask_mc_signal, mask_mc_tag,
    mask_pipi, mask_rhorho, mask_pirho, decay_channel_masks,
)

for directory in ("models", "output", "results"):
    Path(directory).mkdir(parents=True, exist_ok=True)


## 2. Признаки модели


In [2]:
vars = ['cmpmiss', 'missedE', 'ang_taus', "ang_tau0_pmiss", "ang_tau1_pmiss", 
        "p_tau_0_tagcm", "p_tau_1_tagcm", "theta_Btag", 'tau_d_0_0', 'tau_d_1_0', "N_KL",
        'ang_tau1_ROE', 'ang_tau0_ROE', 'idec0', 'idec1']


## 3. Сборка отбора сигнала и фона

Определения физических, MC- и канальных масок находятся в `bdt_header.py`. Здесь задаётся их сочетание для данного BDT.


In [3]:
mask_channel = mask_pipi | mask_rhorho | mask_pirho

# Сигнал: наблюдаемые величины + канал + MC-проверки реконструкции и тега.
mask_sig = mask_physics & mask_channel & mask_mc_signal & mask_mc_tag
# Фон: сохраняем прежнее условие is0 == 1, без MC-проверок сигнальной стороны.
mask_bg = mask_physics & mask_channel & mask_mc_tag


## 4. Загрузка сигнала и фона


In [4]:
sig = ds.dataset(list(Path("new_sig/").glob("*.parquet")), format="parquet")
sig = get_values(sig, vars, filter_mask=mask_sig)
bg = ds.dataset(
    [p for i in [1, 2, 3] for p in Path("gen_data/").glob(f"*{i}.parquet")],
    format="parquet",
)
bg = get_values(bg, vars, filter_mask=mask_bg)


Total data size: 0.47 MB
Total data size: 2.54 MB


In [5]:
bg["label"] = 0
sig["label"] = 1
df = pd.concat([sig, bg], ignore_index=True)


## 5. Разделение train/test


In [6]:
X = df.drop(columns=["label"])
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)


## 6. Обучение BDT


In [7]:
w_train = np.where(y_train == 0, 1.0, np.sum(y_train == 0) / np.sum(y_train == 1))
model = XGBClassifier(
    max_depth=3,
    n_estimators=5000,
    learning_rate=0.02,
    subsample=0.5,
    colsample_bytree=1,
    random_state=42,
)
model.fit(X_train, y_train, sample_weight=w_train)


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,1
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## 7. Качество классификатора


In [8]:
y_pred = model.predict(X_test)
test_pred = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.96      0.89      0.93     10165
           1       0.58      0.82      0.68      1872

    accuracy                           0.88     12037
   macro avg       0.77      0.85      0.80     12037
weighted avg       0.90      0.88      0.89     12037

ROC AUC: 0.9339702599417308
Confusion Matrix:
 [[9061 1104]
 [ 344 1528]]


## 8. Каналы тестовой выборки


In [9]:
test_channel_masks = decay_channel_masks(X_test["idec0"], X_test["idec1"])
channel_masks = {
    name: test_channel_masks[name].to_numpy()
    for name in ("pipi", "rhorho", "pirho")
}
assert np.all(np.sum(np.stack(list(channel_masks.values())), axis=0) == 1), "Каналы должны покрывать выборку без пересечений"


## 9. Распределения отклика BDT


In [ ]:
wbin = 0.03
bins = np.linspace(0, 1, int(1 / wbin + 1))

for cname, cmask in channel_masks.items():
    plt.figure(figsize=(8, 6))
    
    sig_chan = test_pred[(y_test == 1) & cmask]
    bkg_chan = test_pred[(y_test == 0) & cmask]
    counts_sig, _ = np.histogram(sig_chan, bins=bins, density=True)
    errordot(counts_sig, bins=bins, err_func=lambda x: 0, color=PLOT_COLORS["signal"], label="Signal (test)")
    
    counts_bkg, _ = np.histogram(bkg_chan, bins=bins, density=True)
    bin_centers = 0.5*(bins[:-1] + bins[1:])
    plt.bar(
        bin_centers, counts_bkg,
        width=wbin, align='center',
        fill=False, alpha=0.7, linewidth=1,
        edgecolor=PLOT_COLORS["background"], color=PLOT_COLORS["background"],
        label="Background (test)"
    )
    
    plt.xlabel("BDT response")
    plt.ylabel(f'$\\mathrm{{Events}}\\,/\\,{wbin}$')
    plt.title(f"BDT response: {cname}")
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(f"output/BDT_output_{cname}.pdf", dpi=700)
    plt.show()


## 10. Важность признаков


In [ ]:
# Модель обучена на DataFrame: get_score возвращает имена исходных колонок.
imp_dict = model.get_booster().get_score(importance_type="gain")
# idec участвуют в обучении; исключаем их только из этого графика.
imp_filtered = {name: value for name, value in imp_dict.items()
                if name not in {"idec0", "idec1"}}

if not imp_filtered:
    print("[info] После фильтрации не осталось признаков для отображения.")
else:
    s = pd.Series(imp_filtered).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    s.plot(kind='barh', ax=ax, color=PLOT_COLORS["diagnostic"])

    xmax = s.max()
    for i, (feat, val) in enumerate(s.items()):
        ax.text(val + 0.01 * xmax, i, f"{val:.3f}",
                va='center', ha='left', fontsize=10)

    ax.set_title("Feature importance (gain)", fontsize=14)
    ax.set_xlabel("Importance score (gain)", fontsize=12)
    ax.set_ylabel("Features", fontsize=12)
    plt.tight_layout()
    plt.savefig("output/BDT_features_pirho.pdf", dpi=700)
    plt.show()


## 11. Выбор порога по Figure of Merit


In [ ]:
# Формула и нормировка перенесены из BDT_rho_pi_lep.ipynb.
# B/3 и +1 — принятые в эталоне настройки, а не подтверждённая MC-нормировка.
cuts = np.linspace(0.0, 1.0, 50)
a = 3.0
background_divisor = 3.0
background_offset = 1.0
FOM_cuts = {}
FOM_results = {}
FOM_curves = {}

for cname, cmask in channel_masks.items():
    is_sig_c = (y_test.to_numpy() == 1) & cmask
    is_bkg_c = (y_test.to_numpy() == 0) & cmask
    Nsig_tot = int(np.sum(is_sig_c))
    Nbkg_tot = int(np.sum(is_bkg_c))
    if Nsig_tot == 0:
        raise ValueError(f"[{cname}] Нет сигнала для вычисления эффективности и FoM")

    preds_sig_c = test_pred[is_sig_c]
    preds_bkg_c = test_pred[is_bkg_c]
    S = np.array([np.count_nonzero(preds_sig_c > cut) for cut in cuts])
    B_raw = np.array([np.count_nonzero(preds_bkg_c > cut) for cut in cuts])
    effS = S / Nsig_tot
    effB = B_raw / Nbkg_tot if Nbkg_tot else np.zeros_like(cuts)
    B = B_raw + background_offset
    FoM = effS / (a / 2 + np.sqrt(B / background_divisor))

    imax = int(np.argmax(FoM))
    best_cut = float(cuts[imax])
    best_fom = float(FoM[imax])
    best_eff = float(effS[imax])
    FOM_cuts[cname] = best_cut  # Совместимость: потребители используют FoM как порог BDT.
    FOM_results[cname] = {
        "cut": best_cut, "max_fom": best_fom, "signal_efficiency": best_eff,
        "background_efficiency": float(effB[imax]),
        "signal_pass": int(S[imax]), "signal_total": Nsig_tot,
        "background_pass": int(B_raw[imax]), "background_total": Nbkg_tot,
        "background_in_fom": float(B[imax] / background_divisor),
    }
    FOM_curves[cname] = {
        "cuts": cuts.tolist(), "fom": FoM.tolist(),
        "signal_efficiency": effS.tolist(), "background_efficiency": effB.tolist(),
        "background_pass": B_raw.tolist(),
    }
    json_update(f"results/data_{cname}.json", {
        "BDT_eff": best_eff, "BDT_cut": best_cut, "BDT_FoM": best_fom,
    })

    plt.figure(figsize=(10, 6))
    plt.plot(cuts, FoM, color=PLOT_COLORS["diagnostic"], label=f"FoM ({cname})")
    plt.axvline(best_cut, color=PLOT_COLORS["data"], linestyle="--", label=f"Max at {best_cut:.3f}")
    plt.xlabel("BDT output")
    plt.ylabel("FoM")
    plt.title(f"Optimization of classifier cut — {cname}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"output/BDT_FoM_{cname}.pdf", dpi=700)
    plt.show()
    print(f"[{cname}] cut={best_cut:.6f}, FoM={best_fom:.6f}, "
          f"eff={best_eff:.6f}, S={S[imax]}/{Nsig_tot}, B_MC={B_raw[imax]}/{Nbkg_tot}")


## 12. Сохранение модели и порогов


In [13]:
model.save_model("models/bdt_model_pirho.json")
data_to_save = {
    "columns": list(X_train.columns),
    "FoM": FOM_cuts,
    "FoM_results": FOM_results,
    "FoM_curves": FOM_curves,
    "FoM_config": {
        "formula": "epsilon_S / (a/2 + sqrt((B_raw + background_offset)/background_divisor))",
        "a": a, "background_divisor": background_divisor,
        "background_offset": background_offset, "test_fraction": 0.3,
        "background_files": [str(p) for i in [1, 2, 3] for p in sorted(Path("gen_data").glob(f"*{i}.parquet"))],
    },
}
with open("models/columns_and_fom_pirho.json", "w", encoding="utf-8") as f:
    json.dump(data_to_save, f, indent=4, ensure_ascii=False, allow_nan=False)
